# Exercise 1.3.3.7 — verify model + SAEs can still solve this

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.3.3 Interpretability with SAEs`  
**Notebook:** `1.3.3_Interpretability_with_SAEs_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.3.3.7](https://delta-drills.vercel.app/?arena_exercise=1.3.3.7)


# [1.3.3] Interpretability with SAEs (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/13_[1.3.3]_Interpretability_with_SAEs)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part33_interp_with_saes/1.3.3_Interpretability_with_SAEs_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part33_interp_with_saes/1.3.3_Interpretability_with_SAEs_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

> Note - **there is a very large amount of content in this set of exercises**, easily double that of any other single exercise set in ARENA (and some of those exercise sets are meant to last several days). The purpose of these exercises isn't to go through every single one of them, but rather to **jump around to the ones you're most interested in**.
>
> Also, rather than using this material as exercises, you can also just use it as a helpful source of reference code, if you ever want to quickly implement some particular SAE technique or type of forward pass / causal intervention.
>
> You can use the interactive map below to get a better sense of the material content, and the dependencies between different sections.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-13-2.png" width="350">
<br>

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src="https://link.excalidraw.com/readonly/lwbmz3hSOL38907OaCBR", width=1000, height=500))

# Introduction

In these exercises, we dive deeply into the interpretability research that can be done with sparse autoencoders. We jump straight into using SAEs with actual language models, starting by introducing two important tools: `SAELens` (essentially the TransformerLens of SAEs, which also integrates very well with TransformerLens) and **Neuronpedia**, an open platform for interpretability research. We'll then move through a few other exciting domains in SAE interpretability, grouped into several categories (e.g. understanding / classifying latents, training & evaluating SAEs).

We expect some degree of prerequisite knowledge in these exercises. Specifically, it will be very helpful if you understand:

- What **superposition** is
- What the **sparse autoencoder** architecture is, and why it can help us disentangle features from superposition

If you're interested in the theory behind SAEs & superposition, you can go to exercises **1.5.4 Toy Models of SAEs & Superposition** which cover this in detail. In particular, if you speedrun section 1 and the first half of section 5 of that exercise set, this should suffice for understanding the core theory of superposition & SAEs.

One note before starting - we'll be mostly adopting the terminology that **features** are characteristics of the underlying data distribution that our base models are trained on, and **SAE latents** (or just "latents") are the directions in the SAE. This is to avoid the overloading of the term "feature", and avoiding the implicit assumption that "SAE features" correspond to real features in the data. We'll relax this terminology when we're looking at SAE latents which very clearly correspond to specific interpretable features in the data.

## Reading Material

Most of this is optional, and can be read at your leisure depending on what interests you most & what level of background you have. If we could recommend just one, it would be "Towards Monosemanticity" - particularly the first half of "Problem Setup", and the sections where they take a deep dive on individual latents.

- [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) outlines the core ideas behind superposition - what it is, why it matters for interepretability, and what we might be able to do about it.
- [Towards Monosemanticity: Decomposing Language Models With Dictionary Learning](https://transformer-circuits.pub/2023/monosemantic-features/index.html) arguably took the first major stride in mechanistic interpretability with SAEs: training them on a 1-layer model, and extracting a large number of interpretable features.
- [Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html) shows how you can scale up the science of SAEs to larger models, specifically the SOTA (at the time) model Claude 3 Sonnet. It provides an interesting insight into where the field might be moving in the near future.
- [Improving Dictionary Learning with Gated Sparse Autoencoders](https://arxiv.org/pdf/2404.16014) is a paper from DeepMind which introduces the Gated SAE architecture, demonstrating how it outperforms the standard architecture and also motivating its use by speculating about underlying feature distributions.
- [Gemma Scope](https://deepmind.google/discover/blog/gemma-scope-helping-the-safety-community-shed-light-on-the-inner-workings-of-language-models/) announces DeepMind's release of a comprehensive suite of open-sourced SAE models (trained with JumpReLU architecture). We'll be working a lot more with Gemma Scope models in subsequent exercises!
- [LessWrong, SAEs tag](https://www.lesswrong.com/tag/sparse-autoencoders-saes) contains a collection of posts on LessWrong that discuss SAEs, and is a great source of inspiration for further independent research!

## Content & Learning Objectives

View [this interactive map](https://link.excalidraw.com/l/9KwMnW35Xt8/86r7xyurK0g) to see all the material which is complete & still in development, as well as figures showing what you'll create at the end of each section. It contains pretty much all the information below, but in an easier-to-visualize format.

Highlighting a few important things which the map should make clear:

- There's no required order to go through this material in! With the exception of the first few chunks of section 1️⃣ you can basically pick whatever you want to work through, depending on what your Objectives are. The map shows the dependencies between different sections, which you can use to guide your work.
- Some sections of material are still in development, and will continue to be added to over October & November (although development will mostly be paused during the middle of October, while I'm going through an interview process). We're open to suggestions or recommendations for how to improve / add to this material further!

### 1️⃣ Intro to SAE Interpretability

The idea is for this section is to be an MVP for all basic SAE topics, excluding training & evals (which we'll come back to in section 3️⃣). The focus will be on how to understand & interpret SAE latents (in particular all the components of the [SAE dashboard](https://transformer-circuits.pub/2023/monosemantic-features/vis/a1.html)). We'll also look at techniques for finding latents (e.g. ablation & attribution methods), as well as taking a deeper dive into attention SAEs and how they work.

> ##### Learning Objectives
>   
> - Learn how to use the `SAELens` library to load in & run SAEs (alongside the TransformerLens models they're attached to)
> - Understand the basic features of **Neuronpedia**, and how it can be used for things like steering and searching over features
> - Understand **SAE dashboards**, what each part of them tells you about a particular latent (as well as how to compute them yourself)
> - Learn techniques for finding latents, including **direct logit attribution**, **ablation** and **attribution patching**
> - Use **attention SAEs**, understand how they differ from regular SAEs (as well as topics specific to attention SAEs, like **direct latent attribution**)
> - Learn a bit about different SAE architectures or training methods (e.g. gated, end-to-end, meta-saes, transcoders) - some of these will be covered in more detail later

### 2️⃣ Understanding SAE Latents: A Deeper Dive

This is essentially an a-la-carte batch of several different topics, which aren't specifically related to SAE circuits or training SAEs, but which were too niche or in-depth to cover in the intro section. Much of this represents research being done on the cutting edge of current SAE interpretability, and could be an interesting jumping off point for your own research!

> ##### Learning Objectives
>
> - Study feature splitting, and what it means for SAE training
> - Use UMAPs and other dimensionality-reduction techniques to better understand SAE latent geometry
> - Understand feature absorption, and how meta-SAEs might help us disentangle this problem (not implemented yet)
> - Use the logit lens & techniques like token enrichment analysis to better understand & characterize SAE latents (not implemented yet)
> - Take a deeper dive into automated interpretability, covering autointerp-based evals & patch scoping (not implemented yet)

### 3️⃣ Training & Evaluating SAEs

In this section, we first cover some basic material on training SAEs. We'll show you how `SAELens` supports SAE training and cover a few general pieces of advice, and then go through a few training case studies. Each of these training exercises represents a good jumping off point for further investigation of your trained models (although this is more true for the smaller models e.g. the attention SAE trained on a 2L model, since it's more likely to have found features that match the level of complexity of the base model, given that these tutorials are optimised for brevity and low compute requirements!).

We plan to add a second half to this section which covers evals, but this is currently still in development (and we cover many evals-related things in other parts of the material, e.g. the first half of this section, as well as the autointerp material in section 2️⃣).

> ##### Learning Objectives
>
> - Learn how to train SAEs using `SAELens`
> - Understand how to interpret different metrics during training, and understand when & why SAE training fails to produce interpretable latents
> - Get hands-on experience training SAEs in a variety of context: MLP output of TinyStories-1L, residual stream of Gemma-2-2B, attention output of a 2L model, etc
> - Understand how to evaluate SAEs, and why simple metrics can be deceptive (not implemented yet)

## A note on memory usage

In these exercises, we'll be loading some pretty large models into memory (e.g. Gemma 2-2B and its SAEs, as well as a host of other models in later sections of the material). It's useful to have functions which can help profile memory usage for you, so that if you encounter OOM errors you can try and clear out unnecessary models. For example, we've found that with the right memory handling (i.e. deleting models and objects when you're not using them any more) it should be possible to run all the exercises in this material on a Colab Pro notebook, and all the exercises minus the handful involving Gemma on a free Colab notebook.

<details>
<summary>See this dropdown for some functions which you might find helpful, and how to use them.</summary>

First, we can run some code to inspect our current memory usage. Here's me running this code during the exercise set on SAE circuits, after having already loaded in the Gemma models from the previous section. This was on a Colab Pro notebook.

```python
import part33_interp_with_saes.utils as utils

# Profile memory usage, and delete gemma models if we've loaded them in
namespace = globals().copy() | locals()
utils.profile_pytorch_memory(namespace=namespace, filter_device="cuda:0")
```

<pre style="font-family: Consolas; font-size: 14px">Allocated = 35.88 GB
Total = 39.56 GB
Free = 3.68 GB
┌──────────────────────┬────────────────────────┬──────────┬─────────────┐
│ Name                 │ Object                 │ Device   │   Size (GB) │
├──────────────────────┼────────────────────────┼──────────┼─────────────┤
│ gemma_2_2b           │ HookedSAETransformer   │ cuda:0   │       11.94 │
│ gpt2                 │ HookedSAETransformer   │ cuda:0   │        0.61 │
│ gemma_2_2b_sae       │ SAE                    │ cuda:0   │        0.28 │
│ sae_resid_dirs       │ Tensor (4, 24576, 768) │ cuda:0   │        0.28 │
│ gpt2_sae             │ SAE                    │ cuda:0   │        0.14 │
│ logits               │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ logits_with_ablation │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ clean_logits         │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ _                    │ Tensor (16, 128, 768)  │ cuda:0   │        0.01 │
│ clean_sae_acts_post  │ Tensor (4, 15, 24576)  │ cuda:0   │        0.01 │
└──────────────────────┴────────────────────────┴──────────┴─────────────┘</pre>

From this, we see that we've allocated a lot of memory for the the Gemma model, so let's delete it. We'll also run some code to move any remaining objects on the GPU which are larger than 100MB to the CPU, and print the memory status again.

```python
del gemma_2_2b
del gemma_2_2b_sae

THRESHOLD = 0.1  # GB
for obj in gc.get_objects():
    try:
        if isinstance(obj, t.nn.Module) and part32_utils.get_tensors_size(obj) / 1024**3 > THRESHOLD:
            if hasattr(obj, "cuda"):
                obj.cpu()
            if hasattr(obj, "reset"):
                obj.reset()
    except:
        pass

# Move our gpt2 model & SAEs back to GPU (we'll need them for the exercises we're about to do)
gpt2.to(device)
gpt2_saes = {layer: sae.to(device) for layer, sae in gpt2_saes.items()}

part32_utils.print_memory_status()
```

<pre style="font-family: Consolas; font-size: 14px">Allocated = 14.90 GB
Reserved = 39.56 GB
Free = 24.66</pre>

Mission success! We've managed to free up a lot of memory. Note that the code which moves all objects collected by the garbage collector to the CPU is often necessary to free up the memory. We can't just delete the objects directly because PyTorch can still sometimes keep references to them (i.e. their tensors) in memory. In fact, if you add code to the for loop above to print out `obj.shape` when `obj` is a tensor, you'll see that a lot of those tensors are actually Gemma model weights, even once you've deleted `gemma_2_2b`.

</details>

## Setup (don't read, just run)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install "openai==1.56.1" einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" openai tabulate umap-learn hdbscan eindex-callum git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python git+https://github.com/callummcdougall/sae_vis.git@callum/v3 transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import gc
import itertools
import os
import random
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Literal, TypeAlias

import circuitsvis as cv
import einops
import pandas as pd
import plotly.express as px
import requests
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import HTML, IFrame, display
from jaxtyping import Float, Int
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from sae_lens import (
    SAE,
    ActivationsStore,
    GatedTrainingSAEConfig,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    LoggingConfig,
)
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


def _get_hook_layer(sae: SAE) -> int:
    """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
    return int(sae.cfg.metadata.hook_name.split(".")[1])


# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part33_interp_with_saes"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part33_interp_with_saes.tests as tests
import part33_interp_with_saes.utils as utils

MAIN = __name__ == "__main__"

In [ ]:
# For displaying sae-vis inline
if IN_COLAB:
    import http.server
    import socketserver
    import threading

    from google.colab import output

    PORT = 8000

    def display_vis_inline(filename: Path, height: int = 850):
        """
        Displays the HTML files in Colab. Uses global `PORT` variable defined in prev cell, so that each
        vis has a unique port without having to define a port within the function.
        """
        global PORT

        def serve(directory):
            os.chdir(directory)
            handler = http.server.SimpleHTTPRequestHandler
            with socketserver.TCPServer(("", PORT), handler) as httpd:
                print(f"Serving files from {directory} on port {PORT}")
                httpd.serve_forever()

        thread = threading.Thread(target=serve, args=("/content",))
        thread.start()

        filename = str(filename).split("/content")[-1]

        output.serve_kernel_port_as_iframe(PORT, path=filename, height=height, cache_in_notebook=True)

        PORT += 1

# 1️⃣ Intro to SAE Interpretability

> ##### Learning Objectives
>
> - Learn how to use the `SAELens` library to load in & run SAEs (alongside the TransformerLens models they're attached to)
> - Understand the basic features of **Neuronpedia**, and how it can be used for things like steering and searching over features
> - Understand **SAE dashboards**, what each part of them tells you about a particular latent (as well as how to compute them yourself)
> - Learn techniques for finding latents, including **direct logit attribution**, **ablation** and **attribution patching**
> - Use **attention SAEs**, understand how they differ from regular SAEs (as well as topics specific to attention SAEs, like **direct latent attribution**)
> - Learn a bit about different SAE architectures or training methods (e.g. gated, end-to-end, meta-saes, transcoders) - some of these will be covered in more detail later

To emphasize - the idea is for this section is to be a whirlwind tour of all basic SAE topics, excluding training & evals (which we'll come back to in section 4). The focus will be on how to understand & interpret SAE latents (in particular all the components of the [SAE dashboard](https://transformer-circuits.pub/2023/monosemantic-features/vis/a1.html)). We'll also look at techniques for finding latents (e.g. ablation & attribution methods), as well as taking a deeper dive into attention SAEs and how they work. Because there's a lot of material to cover in this section, we'll have a summary of the key points at the top of each main header section. These summaries are all included below for convenience, before we get started. As well as helping to keep you oriented as you work through the material, these should also give you an idea of which sections you can jump to if you only want to cover a few of them.

<details>
<summary>Intro to SAELens</summary>

In this section, you'll learn what `SAELens` is, and how to use it to load in & inspect the configs of various supported SAEs. Key points:

- SAELens is a library for training and analysing SAEs. It can be thought of as the equivalent of TransformerLens for SAEs (although it allso integrates closely with TransformerLens, as we'll see in the "Running SAEs" section)
- SAELens contains many different model releases, each release containing multiple SAEs (e.g. trained on different model layers / hook points, or with different architectures)
- The `cfg` attribute of an `SAE` instance contains this information, and anything else that's relevant when performing forward passes

</details>

<details>
<summary>Visualizing SAEs with dashboards</summary>

In this section, you'll learn about SAE dashboards, which are a visual tool for quickly understanding what a particular SAE latent represents. Key points:

- Neuronpedia hosts dashboards which help you understand SAE latents
- The 5 main components of the dashboard are: top logit tables, logits histogram, activation density plots, top activating sequences, and autointerp
- All of these components are important for getting a full picture of what a latent represents, but they can also all be misleading
- You can display these dashboards inline, using `IFrame`

</details>

<details>
<summary>Running SAEs</summary>

In this section, you'll learn how to run forward passes with SAEs. This is a pretty simple process, which builds on much of the pre-existing infrastructure in TransformerLens models. Key points:

- You can add SAEs to a TransformerLens model when doing forward passes in pretty much the same way you add hook functions (you can think of SAEs as a special kind of hook function)
- When `sae.error_term=False` (default) you substitute the SAE's output for the transformer activations. When True, you don't substitute (which is sometimes what you want when caching activations)
- There's an analogous `run_with_saes` that works like `run_with_hooks`
- There's also `run_with_cache_with_saes` that works like `run_with_cache`, but allows you to cache any SAE activations you want
- You can use `ActivationStore` to get a large batch of activations at once

</details>

<details>
<summary>Replicating SAE dashboards</summary>

In this section, you'll replicate the 5 main components of the SAE dashboard: top logits tables, logits histogram, activation density plots, top activating sequences, and autointerp. There's not really any new content here, just putting into practice what you've learned from the previous 2 sections "Visualizing SAEs with dashboards" and "Running SAEs".

</details>

<details>
<summary>Attention SAEs</summary>

In this section, you'll learn about attention SAEs, how they work (mostly quite similar to standard SAEs but with a few other considerations), and how to understand their feature dashboards. Key points:

- Attention SAEs have the same architecture as regular SAEs, except they're trained on the concatenated pre-projection output of all attention heads.
- If a latent fires on a destination token, we can use **direct latent attribution** to see which source tokens it primarily came from.
- Just like regular SAEs, latents found in different layers of a model are often qualitatively different from each other.

</details>

<details>
<summary>Finding latents for features</summary>

In this section, you'll explore different methods (some causal, some not) for finding latents in SAEs corresponding to particular features. Key points:

- You can look at **max activating latents** on some particular input prompt, this is basically the simplest thing you can do
- **Direct logit attribution (DLA)** is a bit more refined; you can find latents which have a direct effect on specific logits
- **Ablation** of SAE latents can help you find latents which are important in a non-direct way
- ...but it's quite costly for a large number of latents, so you can use **attribution patching** as a cheaper linear approximation of ablation

</details>

<details>
<summary>GemmaScope</summary>

This short section introduces you to DeepMind's GemmaScope series, a suite of highly performant SAEs which can be a great source of study in your own interpretability projects!

</details>

<details>
<summary>Feature steering</summary>

In this section, you'll learn how to steer on latents to produce interesting model output. Key points:

- Steering involves intervening during a forward pass to change the model's activations in the direction of a particular latent
- The steering behaviour is sometimes unpredictable, and not always equivalent to "produce text of the same type as the latent strongly activates on"
- Neuronpedia has a steering interface which allows you to steer without any code

</details>

<details>
<summary>Other types of SAEs</summary>

This section introduces a few different SAE architectures, some of which will be explored in more detail in later sections. There are no exercises here, just brief descriptions. Key points:

- Different activation functions / encoder architecturs e.g. **TopK**, **JumpReLU** and **Gated** models can solve problems like feature suppression and the pressure for SAEs to be continuous in standard models
- **End-to-end SAEs** are trained with a different loss function, encouraging them to learn features that are functionally useful for the model's output rather than just minimising MSE reconstruction error
- **Transcoders** are a type of SAE which learn to reconstruct a model's computation (e.g. a sparse mapping from MLP input to MLP output) rather than just reconstructing activations; they can sometimes lead to easier circuit analysis

</details>

## Intro to SAELens

> In this section, you'll learn what `SAELens` is, and how to use it to load in & inspect the configs of various supported SAEs. Key points:
>
> - SAELens is a library for training and analysing SAEs. It can be thought of as the equivalent of TransformerLens for SAEs (although it allso integrates closely with TransformerLens, as we'll see in the "Running SAEs" section)
> - SAELens contains many different model releases, each release containing multiple SAEs (e.g. trained on different model layers / hook points, or with different architectures)
> - The `cfg` attribute of an `SAE` instance contains this information, and anything else that's relevant when performing forward passes

[SAELens](https://github.com/jbloomAus/SAELens) is a library designed to help researchers:

- Train sparse autoencoders,
- Analyse sparse autoencoders / research mechanistic interpretability,
- Generate insights which make it easier to create safe and aligned AI systems.

You can think of it as the equivalent of TransformerLens for sparse autoencoders (and it also integrates very well with TransformerLens models, which we'll see shortly).

Additionally, SAELens is closely integrated with [Neuronpedia](https://neuronpedia.org), an open platform for interpretability research developed through the joint efforts of Joseph Bloom and Johnny Lin, and which we'll be using throughout this chapter. Neuronpedia allows you to search over latents, run SAEs, and even upload your own SAEs!

Before we load our SAEs, it can be useful to see which are available. The following snippet shows the currently available SAE releases in SAELens, and will remain up-to-date as SAELens continues to add more SAEs.

In [ ]:
print(get_pretrained_saes_directory())

Let's print out all this data in a more readable format, with only a subset of attributes. We'll look at `model` (the base model), `release` (the name of the SAE release), `repo_id` (the id of the HuggingFace repo containing the SAEs), and also the number of SAEs in each release (e.g. a release might contain an SAE trained on each layer of the base model).

In [ ]:
metadata_rows = [
    [data.model, data.release, data.repo_id, len(data.saes_map)] for data in get_pretrained_saes_directory().values()
]

# Print all SAE releases, sorted by base model
print(
    tabulate(
        sorted(metadata_rows, key=lambda x: x[0]),
        headers=["model", "release", "repo_id", "n_saes"],
        tablefmt="simple_outline",
    )
)

Any given SAE release may have multiple different mdoels. These might have been trained on different hookpoints or layers in the model, or with different hyperparameters, etc. You can see the data associated with each release as follows:

In [ ]:
def format_value(value):
    return "{{{0!r}: {1!r}, ...}}".format(*next(iter(value.items()))) if isinstance(value, dict) else repr(value)


release = get_pretrained_saes_directory()["gpt2-small-res-jb"]

print(
    tabulate(
        [[k, format_value(v)] for k, v in release.__dict__.items()],
        headers=["Field", "Value"],
        tablefmt="simple_outline",
    )
)

Let's get some more info about each of the SAEs associated with each release. We can print out the SAE id, the path (i.e. in the HuggingFace repo, which points to the SAE model weights) and the Neuronpedia ID (which is how we'll get feature dashboards - more on this soon).

In [ ]:
data = [[id, path, release.neuronpedia_id[id]] for id, path in release.saes_map.items()]

print(
    tabulate(
        data,
        headers=["SAE id", "SAE path (HuggingFace)", "Neuronpedia ID"],
        tablefmt="simple_outline",
    )
)

Next, we'll load the SAE which we'll be working with for most of these exercises: the **layer 7 resid pre model** from the **GPT2 Small SAEs** (as well as a copy of GPT2 Small to attach it to). The SAE uses the `HookedSAETransformer` class, which is adapted from the TransformerLens `HookedTransformer` class.

Note, the `SAE.from_pretrained` function returns an SAE object directly. The SAE's config and metadata can be accessed via `sae.cfg` and `sae.cfg.metadata` respectively.

In [ ]:
t.set_grad_enabled(False)

gpt2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gpt2-small", device=device)

gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=str(device),
)

The `sae` object is an instance of the `SAE` (Sparse Autoencoder) class. There are many different SAE architectures which may have different weights or activation functions. In order to simplify working with SAEs, SAELens handles most of this complexity for you. You can run the cell below to see each of the SAE config parameters for the one we'll be using.

<details>
<summary>Click to read a description of each of the SAE config parameters.</summary>

1. `architecture`: Specifies the type of SAE architecture being used, in this case, the standard architecture (encoder and decoder with hidden activations, as opposed to a gated SAE).
2. `d_in`: Defines the input dimension of the SAE, which is 768 in this configuration.
3. `d_sae`: Sets the dimension of the SAE's hidden layer, which is 24576 here. This represents the number of possible feature activations.
4. `activation_fn_str`: Specifies the activation function used in the SAE, which is ReLU in this case. TopK is another option that we will not cover here.
5. `apply_b_dec_to_input`: Determines whether to apply the decoder bias to the input, set to True here.
6. `finetuning_scaling_factor`: Indicates whether to use a scaling factor to weight initialization and the forward pass. This is not usually used and was introduced to support a [solution for shrinkage](https://www.lesswrong.com/posts/3JuSjTZyMzaSeTxKk/addressing-feature-suppression-in-saes).
7. `context_size`: Defines the size of the context window, which is 128 tokens in this case. In turns out SAEs trained on small activations from small prompts [often don't perform well on longer prompts](https://www.lesswrong.com/posts/baJyjpktzmcmRfosq/stitching-saes-of-different-sizes).
8. `model_name`: Specifies the name of the model being used, which is 'gpt2-small' here. [This is a valid model name in TransformerLens](https://transformerlensorg.github.io/TransformerLens/generated/model_properties_table.html).
9. `hook_name`: Indicates the specific hook in the model where the SAE is applied.
10. `hook_layer`: Specifies the layer number where the hook is applied, which is layer 7 in this case.
11. `hook_head_index`: Defines which attention head to hook into; not relevant here since we are looking at a residual stream SAE.
12. `prepend_bos`: Determines whether to prepend the beginning-of-sequence token, set to True.
13. `dataset_path`: Specifies the path to the dataset used for training or evaluation. (Can be local or a huggingface dataset.)
14. `dataset_trust_remote_code`: Indicates whether to trust remote code (from HuggingFace) when loading the dataset, set to True.
15. `normalize_activations`: Specifies how to normalize activations, set to 'none' in this config.
16. `dtype`: Defines the data type for tensor operations, set to 32-bit floating point.
17. `device`: Specifies the computational device to use.
18. `sae_lens_training_version`: Indicates the version of SAE Lens used for training, set to None here.
19. `activation_fn_kwargs`: Allows for additional keyword arguments for the activation function. This would be used if e.g. the `activation_fn_str` was set to `topk`, so that `k` could be specified.

</details>

In [ ]:
print(
    tabulate(
        list(gpt2_sae.cfg.__dict__.items()) + list(gpt2_sae.cfg.metadata.items()),
        headers=["name", "value"],
        tablefmt="simple_outline",
    )
)

## Visualizing SAEs with dashboards

> In this section, you'll learn about SAE dashboards, which are a visual tool for quickly understanding what a particular SAE latent represents. Key points:
>
> - Neuronpedia hosts dashboards which help you understand SAE latents
> - The 5 main components of the dashboard are: top logit tables, logits histogram, activation density plots, top activating sequences, and autointerp
> - All of these components are important for getting a full picture of what a latent represents, but they can also all be misleading
> - You can display these dashboards inline, using `IFrame`

In this section, we're going to have a look at our SAEs, and see what they're actually telling us.

Before we dive too deep however, let's recap something - what actually is an SAE latent?

An SAE latent is a particular direction** in the base model's activation space, learned by the SAE. Often, these correspond to **features** in the data - in other words, meaningful semantic, syntactic or otherwise interpretable patterns or concepts that exist in the distribution of data the base model was trained on, and which were learned by the base model. These features are usually highly sparse, in other words for any given feature only a small fraction of the overall data distribution will activate that feature. It tends to be the case that sparser features are also more interpretable.

**Note - technically saying "direction" is an oversimplification here, because a given latent can have multiple directions in activation space associated with them, e.g. a separate encoder and decoder direction for standard untied SAEs. When we refer to a latent direction or feature direction, we're usually but not always referring to the decoder weights.

The dashboard shown below provides a detailed view of a single SAE latent.

In [ ]:
def display_dashboard(
    sae_release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    latent_idx=0,
    width=800,
    height=600,
):
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

    print(url)
    display(IFrame(url, width=width, height=height))


latent_idx = random.randint(0, gpt2_sae.cfg.d_sae)
display_dashboard(latent_idx=latent_idx)

Let's break down the separate components of the visualization:

1. **Latent Activation Distribution**. This shows the proportion of tokens a latent fires on, usually between 0.01% and 1%, and also shows the distribution of positive activations.  
2. **Logits Distribution**. This is the projection of the decoder weight onto the unembed and roughly gives us a sense of the tokens promoted by a latent. It's less useful in big models / middle layers.
3. **Top / Botomn Logits**. These are the 10 most positive and most negative logits in the logit weight distribution.
4. **Max Activating Examples**. These are examples of text where the latent fires and usually provide the most information for helping us work out what a latent means.
5. **Autointerp**. These are LLM-generated latent explanations, which use the rest of the data in the dashboard (in particular the max activating examples).

See this section of [Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features#setup-interface) for more information.

*Neuronpedia* is a website that hosts SAE dashboards and which runs servers that can run the model and check latent activations. This makes it very convenient to check that a latent fires on the distribution of text you actually think it should fire on. We've been downloading data from Neuronpedia for the dashboards above.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.3.3.7"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part33_interp_with_saes.solutions import show_activation_histogram, get_k_largest_indices, show_top_logits, get_autointerp_df, display_top_seqs_attn, display_top_seqs_attn


### Exercise - verify model + SAEs can still solve this

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

We want to make sure that the logit diff for the model is still high when we substitute in SAEs for a given layer. If this isn't the case, then that means our SAEs aren't able to implement the IOI circuit, and there's no reason to expect we'll find any interesting features!

You should use the `with model.saes` context manager (or whatever your preferred way of running with SAEs is) to get the average logit diff over the 4 prompts, for each layer's attention SAE. Verify that this difference is still high, i.e. the SAEs don't ruin performance.

In [ ]:
# YOUR CODE HERE - verify model + SAEs can still solve this

<details><summary>Solution</summary>

```python
logits = gpt2(prompts, return_type="logits")
clean_logit_diff = logits_to_ave_logit_diff(logits)

table = Table("Ablation", "Logit diff", "% of clean")

table.add_row("Clean", f"{clean_logit_diff:+.3f}", "100.0%")

for layer in range(gpt2.cfg.n_layers):
    with gpt2.saes(saes=[attn_saes[layer]]):
        logits = gpt2(prompts, return_type="logits")
        logit_diff = logits_to_ave_logit_diff(logits)
        table.add_row(
            f"SAE in L{layer:02}",
            f"{logit_diff:+.3f}",
            f"{logit_diff / clean_logit_diff:.1%}",
        )

rprint(table)
```
</details>

<details>
<summary>Discussion of results</summary>

You should find that most layers have close to 100% recovery, even some layers like 10 and 11 which *increase* the logit diff when they're substituted in. You shouldn't read too much into that in this case, because we're only working with a small dataset and so some noise is expected.

The worst layers in terms of logit diff recovery are 7 and 8, which makes sense given that these are the layers with our S-Inhibition heads. The fact that the logit diff is still positive for both, and remains positive when you substitute in both layer 7 and 8 SAEs at once (although it does drop to 58% of the clean logit diff) means that these SAEs are still presumably capturing some amount of the S-Inhibition heads' behaviour - however that doesn't guarantee we have monosemantic S-Inhibition features, which in fact we don't.

</details>

Now we've done this, it's time for exercises!

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
